# 06 — Unified Results

Loads raw prediction arrays from all evaluation notebooks and recomputes
all metrics uniformly. No dependency on pre-saved metric files.

## Sources
- `raw_preds_flant5.json` — from `02_eval_flant5.ipynb`
- `raw_preds_gpt41mini.json` — from `03_eval_gpt41mini.ipynb`
- `raw_preds_gpt5.json` — from `05_gpt5_results.ipynb`

## Prerequisites
Run notebooks 02, 03, 05 first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, classification_report)

ROOT        = '/content/drive/MyDrive/paper_codes/paper2'
RESULTS_DIR = f'{ROOT}/results'

DOMAINS    = ['blocksworld', 'blocksworld_3', 'mystery', 'mystery_3', 'logistics']
DOM_SHORT  = ['bw', 'bw3', 'mys', 'mys3', 'log']
DOM_MAP    = dict(zip(DOMAINS, DOM_SHORT))

DIRECT_CONDS   = ['FlanT5_Direct', 'FlanT5_Direct_FT50', 'FlanT5_Direct_FT_Random',
                  'G_direct', 'G5_direct']
STEPWISE_CONDS = ['FlanT5_Stepwise', 'FlanT5_Stepwise_FT_FOLIO',
                  'FlanT5_Stepwise_FT_PB', 'FlanT5_Stepwise_FT_Combined',
                  'G_stepwise', 'G5_stepwise']

print(f'RESULTS_DIR: {RESULTS_DIR}')
print('Files:', sorted(os.listdir(RESULTS_DIR)))

## 1. Load Raw Predictions

In [ ]:
raw_all = {}

for fname in ['raw_preds_flant5.json',
              'raw_preds_gpt41mini.json',
              'raw_preds_gpt5.json']:
    path = os.path.join(RESULTS_DIR, fname)
    if os.path.exists(path):
        data = json.load(open(path))
        raw_all.update(data)
        print(f'Loaded {fname}: {list(data.keys())}')
    else:
        print(f'MISSING: {fname}')

def get_arrays(cond, domain):
    """
    Get prediction arrays for a condition/domain.
    Handles key name variants across notebooks:
      failing_step_golds / fail_step_golds  (both used in different notebooks)
      failing_step_preds / fail_step_preds
    Returns dict with normalised keys, empty lists if missing.
    """
    r = raw_all.get(cond, {}).get(domain, {})
    pg  = r.get('plan_golds',   [])
    pp  = r.get('plan_preds',   [])
    ag  = r.get('action_golds', [])
    ap  = r.get('action_preds', [])
    # Failing step — try both key name variants
    fsg = (r.get('fail_step_golds') or
           r.get('failing_step_golds') or [])
    fsp = (r.get('fail_step_preds') or
           r.get('failing_step_preds') or [])
    return {'pg': pg, 'pp': pp, 'ag': ag, 'ap': ap, 'fsg': fsg, 'fsp': fsp}

print()
print('Conditions loaded:', list(raw_all.keys()))

## 2. Compute Metrics

In [ ]:
def compute(pg, pp, ag, ap, fsg, fsp):
    """Compute all metrics from raw prediction arrays."""
    if not pg:
        return None

    plan_f1  = f1_score(pg, pp, average='macro', zero_division=0)
    plan_acc = accuracy_score(pg, pp)
    plan_cm  = confusion_matrix(pg, pp, labels=[0, 1]).tolist()

    act_f1 = act_acc = act_cm = None
    if ag:
        act_f1  = f1_score(ag, ap, average='macro', zero_division=0)
        act_acc = accuracy_score(ag, ap)
        act_cm  = confusion_matrix(ag, ap, labels=[0, 1]).tolist()

    fs_acc = None
    if fsg:
        fs_acc = sum(g == p for g, p in zip(fsg, fsp)) / len(fsg)

    return {
        'n_plans'          : len(pg),
        'n_actions'        : len(ag),
        'n_fs_evaluated'   : len(fsg),
        'plan_f1'          : round(plan_f1,  4),
        'plan_acc'         : round(plan_acc, 4),
        'plan_cm'          : plan_cm,
        'action_f1'        : round(act_f1,  4) if act_f1  is not None else None,
        'action_acc'       : round(act_acc, 4) if act_acc is not None else None,
        'action_cm'        : act_cm,
        'failing_step_acc' : round(fs_acc,  4) if fs_acc  is not None else None,
        'plan_golds'       : pg,  'plan_preds'  : pp,
        'action_golds'     : ag,  'action_preds': ap,
        'fail_step_golds'  : fsg, 'fail_step_preds': fsp,
    }

# Compute metrics for every condition and domain
all_results = {}
for cond in DIRECT_CONDS + STEPWISE_CONDS:
    if cond not in raw_all:
        continue
    all_results[cond] = {}
    for domain in DOMAINS:
        arrays = get_arrays(cond, domain)
        r = compute(**arrays)
        if r:
            all_results[cond][domain] = r

print('Metrics computed for:')
for cond, dom_res in all_results.items():
    domains_done = list(dom_res.keys())
    fs_done = [d for d in domains_done
               if all_results[cond][d]['failing_step_acc'] is not None]
    print(f'  {cond:<40s} domains={domains_done}  fs_acc_domains={fs_done}')

## 3. Summary Tables

In [ ]:
SEP  = '-' * 92
SEP2 = '=' * 92

def print_table(title, cond_names, metric):
    active = [c for c in cond_names
              if c in all_results and
              any(all_results[c].get(d, {}).get(metric) is not None
                  for d in DOMAINS)]
    if not active:
        return
    print(f'\n{SEP}')
    print(title)
    print(f'{"Condition":<40s}' +
          ''.join(f'{s:>8s}' for s in DOM_SHORT) +
          f'  {"Mean":>6s}')
    print(SEP)
    for cond in active:
        vals  = [all_results[cond].get(d, {}).get(metric) for d in DOMAINS]
        valid = [v for v in vals if v is not None]
        mean  = np.mean(valid) if valid else float('nan')
        row   = f'{cond:<40s}' + ''.join(
            f'{v:>8.4f}' if v is not None else f'{"N/A":>8s}'
            for v in vals)
        print(row + f'  {mean:>6.4f}')

print(SEP2)
print('PLAN F1-MACRO')
print(SEP2)
print_table('Direct',   DIRECT_CONDS,   'plan_f1')
print_table('Stepwise', STEPWISE_CONDS, 'plan_f1')

print()
print(SEP2)
print('PLAN ACCURACY')
print(SEP2)
print_table('Direct',   DIRECT_CONDS,   'plan_acc')
print_table('Stepwise', STEPWISE_CONDS, 'plan_acc')

print()
print(SEP2)
print('ACTION F1-MACRO')
print(SEP2)
print_table('Direct',   DIRECT_CONDS,   'action_f1')
print_table('Stepwise', STEPWISE_CONDS, 'action_f1')

print()
print(SEP2)
print('ACTION ACCURACY')
print(SEP2)
print_table('Direct',   DIRECT_CONDS,   'action_acc')
print_table('Stepwise', STEPWISE_CONDS, 'action_acc')

print()
print(SEP2)
print('FAILING STEP ACCURACY')
print(SEP2)
print_table('Direct',   DIRECT_CONDS,   'failing_step_acc')
print_table('Stepwise', STEPWISE_CONDS, 'failing_step_acc')

print()
print('NOTE: FT_Random evaluated on own 20% holdout — not directly comparable.')

## 4. Plan-Level Confusion Matrices

In [ ]:
SEP3 = '=' * 74

def print_plan_cm(cond_names):
    for cond in cond_names:
        if cond not in all_results:
            continue
        print(f'\n{SEP3}')
        print(f'PLAN CM — {cond}')
        print(f'{SEP3}')
        print(f'  {"Domain":<15s}  {"TP(A→A)":>9s}  {"FN(A→B)":>9s}'
              f'  {"FP(B→A)":>9s}  {"TN(B→B)":>9s}'
              f'  {"Prec":>7s}  {"Rec":>7s}  {"F1":>7s}')
        print('  ' + '-' * 82)
        for domain in DOMAINS:
            r = all_results[cond].get(domain)
            if not r:
                continue
            cm = r['plan_cm']
            tn, fp = cm[0][0], cm[0][1]
            fn, tp = cm[1][0], cm[1][1]
            prec = tp / max(1, tp + fp)
            rec  = tp / max(1, tp + fn)
            f1   = 2 * prec * rec / max(1e-9, prec + rec)
            print(f'  {domain:<15s}  {tp:>9d}  {fn:>9d}'
                  f'  {fp:>9d}  {tn:>9d}'
                  f'  {prec:>7.4f}  {rec:>7.4f}  {f1:>7.4f}')

print('DIRECT CONDITIONS')
print_plan_cm(DIRECT_CONDS)
print()
print('STEPWISE CONDITIONS')
print_plan_cm(STEPWISE_CONDS)

## 5. Action-Level Confusion Matrices

Columns: TP(A→A) FN(A→B) FP(B→A) TN(B→B)
- A-recall = TP/(TP+FN): valid steps correctly passed
- B-recall = TN/(TN+FP): invalid steps correctly caught

In [ ]:
for cond in DIRECT_CONDS + STEPWISE_CONDS:
    if cond not in all_results:
        continue
    has_actions = any(
        all_results[cond][d].get('action_f1') is not None
        for d in DOMAINS if d in all_results[cond]
    )
    if not has_actions:
        continue

    print(f'\n{SEP3}')
    print(f'ACTION CM — {cond}')
    print(f'{SEP3}')
    print(f'  {"Domain":<15s}  {"TP(A→A)":>9s}  {"FN(A→B)":>9s}'
          f'  {"FP(B→A)":>9s}  {"TN(B→B)":>9s}'
          f'  {"F1mac":>6s}  {"A-rec":>6s}  {"B-rec":>6s}')
    print('  ' + '-' * 86)

    all_ag, all_ap = [], []
    for domain in DOMAINS:
        r  = all_results[cond].get(domain, {})
        ag = r.get('action_golds', [])
        ap = r.get('action_preds', [])
        if not ag:
            continue
        all_ag.extend(ag)
        all_ap.extend(ap)
        cm = confusion_matrix(ag, ap, labels=[0, 1])
        tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
        f1    = f1_score(ag, ap, average='macro', zero_division=0)
        a_rec = tp / max(1, tp + fn)
        b_rec = tn / max(1, tn + fp)
        print(f'  {domain:<15s}  {tp:>9d}  {fn:>9d}'
              f'  {fp:>9d}  {tn:>9d}'
              f'  {f1:>6.4f}  {a_rec:>6.4f}  {b_rec:>6.4f}')

    if all_ag:
        cm_all = confusion_matrix(all_ag, all_ap, labels=[0, 1])
        tn,fp,fn,tp = cm_all[0,0],cm_all[0,1],cm_all[1,0],cm_all[1,1]
        f1_all = f1_score(all_ag, all_ap, average='macro', zero_division=0)
        a_rec  = tp / max(1, tp + fn)
        b_rec  = tn / max(1, tn + fp)
        print('  ' + '-' * 86)
        print(f'  {"OVERALL":<15s}  {tp:>9d}  {fn:>9d}'
              f'  {fp:>9d}  {tn:>9d}'
              f'  {f1_all:>6.4f}  {a_rec:>6.4f}  {b_rec:>6.4f}')

## 6. Grouped Results: Non-Abstract vs Abstract Domains

In [ ]:
NON_ABSTRACT = ['blocksworld', 'blocksworld_3', 'logistics']
ABSTRACT     = ['mystery', 'mystery_3']

METRICS = [
    ('plan_f1',          'Plan F1-macro'),
    ('plan_acc',         'Plan Accuracy'),
    ('action_f1',        'Action F1-macro'),
    ('action_acc',       'Action Accuracy'),
    ('failing_step_acc', 'Failing Step Accuracy'),
]

def group_mean(cond, domains, metric):
    vals = [all_results[cond].get(d, {}).get(metric)
            for d in domains if cond in all_results]
    vals = [v for v in vals if v is not None]
    return np.mean(vals) if vals else None

def print_grouped(cond_names, group_label):
    active = [c for c in cond_names if c in all_results]
    if not active:
        return
    print(f'\n{"-"*82}')
    print(f'{group_label}')
    print(f'{"-"*82}')
    for metric, label in METRICS:
        has_any = any(
            all_results.get(c, {}).get(d, {}).get(metric) is not None
            for c in active for d in DOMAINS
        )
        if not has_any:
            continue
        print(f'\n  {label}')
        print(f'  {"Condition":<40s}  {"Non-Abstract":>13s}  {"Abstract":>10s}')
        print('  ' + '-' * 67)
        for cond in active:
            na = group_mean(cond, NON_ABSTRACT, metric)
            ab = group_mean(cond, ABSTRACT,     metric)
            na_str = f'{na:.4f}' if na is not None else '   N/A'
            ab_str = f'{ab:.4f}' if ab is not None else '   N/A'
            print(f'  {cond:<40s}  {na_str:>13s}  {ab_str:>10s}')

print('=' * 82)
print('NON-ABSTRACT (bw, bw3, log)  vs  ABSTRACT (mys, mys3)')
print_grouped(DIRECT_CONDS,   'DIRECT CONDITIONS')
print_grouped(STEPWISE_CONDS, 'STEPWISE CONDITIONS')